<h1>Mindestanforderungen 6 - Klassifikation</h1>

In [13]:
import pandas as pd
import numpy as np


from sklearn.datasets import fetch_20newsgroups

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC

from sklearn.pipeline import Pipeline

from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict

<h2>Daten einlesen</h2>

In [ ]:
df = pd.read_csv("stack-overflow-developer-survey-2025/survey_results_public_without_SO.csv")

<h2>Daten aufteilen in Train und Test Split</h3>

In [ ]:
#y = df["JobSat"].fillna("Unknown").astype(str)
df = df.dropna(subset=["JobSat"])

job_sat_num = df["JobSat"].astype(float)

def map_job_sat(x):
    if x <= 3:
        return "Low"
    elif x <= 6:
        return "Medium"
    else:
        return "High"

y = job_sat_num.apply(map_job_sat)


text_cols = df.select_dtypes(include=["object"]).columns.tolist()
X = df[text_cols].fillna("").agg(" ".join, axis=1)

print("X shape:", getattr(X, "shape", None), "len(X):", len(X))
print("y shape:", getattr(y, "shape", None), "len(y):", len(y))


# Train / Temp Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X shape: (26106,) len(X): 26106
y shape: (26106,) len(y): 26106


In [8]:
pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(
        max_df=0.5,
        analyzer="word"
    )),
    ("classifier", LinearSVC())
])

pipeline

,steps,"[('vectorizer', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


<h2>GridSearchCV, um optimale Parameter herauszufinden</h2>

In [11]:
from sklearn.model_selection import GridSearchCV

parameters = {
    'vectorizer__max_df': (0.5, 0.75, 1.0),
    'vectorizer__analyzer': ('word', 'char'),
    'classifier__C': (0.1, 1, 10)
}

gs = GridSearchCV(pipeline, parameters, cv=3, n_jobs=-1)
gs.fit(X_train, y_train)

print("Best Score:", gs.best_score_)
print("Best Params:", gs.best_params_)

Best Score: 0.716148960360783
Best Params: {'classifier__C': 0.1, 'vectorizer__analyzer': 'word', 'vectorizer__max_df': 0.5}


In [9]:
pipeline.fit(X_train, y_train)

,steps,"[('vectorizer', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [10]:
y_pred_test = pipeline.predict(X_test)
print(classification_report(y_test, y_pred_test))

              precision    recall  f1-score   support

        High       0.74      0.95      0.83      3813
         Low       0.32      0.02      0.04       331
      Medium       0.34      0.11      0.16      1185

    accuracy                           0.71      5329
   macro avg       0.47      0.36      0.34      5329
weighted avg       0.62      0.71      0.63      5329



In [15]:
vectorizer = TfidfVectorizer()
feature_selection = SelectFromModel (LinearSVC(penalty="l1", dual=False))
classifier = LinearSVC()

pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("feature_selection", feature_selection),
    ("classifier", classifier)
])

parameters = {
    'vectorizer__max_df': (0.5, 0.75, 1.0),
    'vectorizer__analyzer': ('word', 'char'),
    'feature_selection__threshold': (None, 'mean')
    #'classifier__kernel': ('linear', 'rbf')
}

grid_search = GridSearchCV(pipeline, param_grid=parameters, verbose=10)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5; 1/12] START feature_selection__threshold=None, vectorizer__analyzer=word, vectorizer__max_df=0.5


/Users/jonas/Documents/DataAnalytics/Project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 1/5; 1/12] END feature_selection__threshold=None, vectorizer__analyzer=word, vectorizer__max_df=0.5;, score=0.709 total time=   7.1s
[CV 2/5; 1/12] START feature_selection__threshold=None, vectorizer__analyzer=word, vectorizer__max_df=0.5


/Users/jonas/Documents/DataAnalytics/Project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 2/5; 1/12] END feature_selection__threshold=None, vectorizer__analyzer=word, vectorizer__max_df=0.5;, score=0.712 total time=   7.2s
[CV 3/5; 1/12] START feature_selection__threshold=None, vectorizer__analyzer=word, vectorizer__max_df=0.5


...

[CV 4/5; 12/12] END feature_selection__threshold=mean, vectorizer__analyzer=char, vectorizer__max_df=1.0;, score=0.715 total time=   4.9s
[CV 5/5; 12/12] START feature_selection__threshold=mean, vectorizer__analyzer=char, vectorizer__max_df=1.0


/Users/jonas/Documents/DataAnalytics/Project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV 5/5; 12/12] END feature_selection__threshold=mean, vectorizer__analyzer=char, vectorizer__max_df=1.0;, score=0.716 total time=   4.8s


/Users/jonas/Documents/DataAnalytics/Project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


,estimator,Pipeline(step...LinearSVC())])
,param_grid,"{'feature_selection__threshold': (None, ...), 'vectorizer__analyzer': ('word', ...), 'vectorizer__max_df': (0.5, ...)}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,10
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [16]:
print(grid_search.best_estimator_)

Pipeline(steps=[('vectorizer', TfidfVectorizer()),
                ('feature_selection',
                 SelectFromModel(estimator=LinearSVC(dual=False, penalty='l1'),
                                 threshold='mean')),
                ('classifier', LinearSVC())])


In [18]:
final_pipeline = grid_search.best_estimator_

final_pipeline.fit(X_train, y_train)
print("Default Score des Klassifizieres: Accuracy=", final_pipeline.score(X_test, y_test))

test_labels = final_pipeline.predict(X_test)
print(classification_report(y_test, test_labels))

/Users/jonas/Documents/DataAnalytics/Project/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Default Score des Klassifizieres: Accuracy= 0.7153312066053669
              precision    recall  f1-score   support

        High       0.73      0.97      0.83      3813
         Low       0.29      0.01      0.01       331
      Medium       0.41      0.10      0.16      1185

    accuracy                           0.72      5329
   macro avg       0.48      0.36      0.34      5329
weighted avg       0.63      0.72      0.63      5329

